Copyright (c) 2026, NVIDIA CORPORATION. Licensed under the Apache License, Version 2.0 (the "License") you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.


# Using MolMIM Embeddings to Cluster Molecules

## What You Will Do

- Call the MolMIM `/hidden` endpoint to obtain molecular embeddings.
- Cluster a small molecule panel using those hidden-state vectors.
- Interpret clustering as a qualitative way to inspect molecular similarity in learned latent space.


```{note}
This notebook reads the MolMIM endpoint from `MOLMIM_URL`, or from `.openhackathon-nims.env` when the service wrapper generated one. If you have not started the NIM services yet, follow `00_Container_Setup.ipynb` first.
```

Generative AI models, such as MolMIM, can be trained on molecular structures to learn a representation of molecules, often referred to as a "molecular embedding". These embeddings are high-dimensional vectors that capture the essential features and properties of molecules.

Once trained, these models can be used to generate embeddings for new, unseen molecules. These embeddings can then be clustered using various clustering algorithms, such as k-means, hierarchical clustering, or density-based clustering.

**Significance of clustering molecular embeddings:**

Clustering molecular embeddings can reveal meaningful patterns and relationships between molecules that are not immediately apparent from their chemical structures. By grouping molecules with similar properties and behaviors, clustering can:

1. **Identify molecular families**: Clustering can help identify groups of molecules with similar chemical structures, properties, and biological activities, which can be useful for understanding the relationships between molecular structure and function.
2. **Reveal functional relationships**: Clustering can uncover functional relationships between molecules, such as molecules with similar biological targets or mechanisms of action.
3. **Improve molecular property prediction**: By identifying clusters of molecules with similar properties, clustering can improve the accuracy of molecular property prediction models, such as those used for toxicity prediction or drug design.
4. **Guide lead optimization**: Clustering can help identify clusters of molecules with similar biological activities, which can guide lead optimization efforts and accelerate the discovery of new drugs.
5. **Uncover novel molecular mechanisms**: Clustering can reveal novel molecular mechanisms or biological pathways by identifying clusters of molecules with similar biological activities or properties.

In the following notebook, we provide an example using the MolMIM NIM to generate embeddings for a small set of molecules. We then use those embeddings to cluster the molecules hierarchically.

We start by importing the necessary libraries, including `requests` for making HTTP requests, `json` for working with JSON data, `pandas` for data manipulation, `numpy` for numerical computations, `seaborn` for data visualization, and `matplotlib` for plotting.

In [ ]:
import os
import requests
import json
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

The next block defines the URL and headers for making an HTTP request to the MolMIM NIM `hidden` endpoint. Source `.openhackathon-nims.env` or set `MOLMIM_URL` if your assigned endpoint differs.


In [ ]:
import sys
from pathlib import Path
# Define the URL and headers
CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD if (CWD / "scoring").exists() else CWD.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
try:
    from scoring.endpoint_env import load_openhackathon_env
    load_openhackathon_env()
except Exception:
    pass

molmim_url = os.environ.get("MOLMIM_URL", "http://localhost:8001").rstrip("/")
url = f"{molmim_url}/hidden"
headers = {
    "accept": "application/json",
    "Content-Type": "application/json",
}
print(f"MolMIM endpoint: {molmim_url}")


This block defines a list of molecule sequences in SMILES format. These sequences will be used to generate embeddings, which are numerical representations of the molecules. The molecules here are a very small subset of those found in the ZINC15 database.

In [ ]:
molecules = [
    "CC[C@@H](OC)[C@H](Cl)C(=O)N1CCCCC1",
    "C[C@H]([C@@H](C)O)S(=O)(=O)CC(=O)N1CCC1",
    "CCOC(=O)C1CCN(C(=O)[C@@H](N)C[C@H](C)CC)CC1",
    "CCN1CCN(C(=O)[C@H](C)N2C[C@@H](C)[S@+]([O-])[C@@H](C)C2)CC1",
    "C[C@H](CC(N)=O)C(=O)N1CCCN(C(=O)CN2CCC[C@H]2C)CC1",
    "CCC1(NCCC[C@H](C)O)CCN(C(=O)CC2(O)CCC2)CC1",
    "CCOC(=O)[C@@H]1CCCN(C(=O)CN(C)[C@@H](C)C(=O)N2CCC[C@@H](C(=O)OCC)C2)C1",
    "NC(=O)C[C@@H](N)C(=O)N1CCN(C(=O)CN2CCCC2)CC1",
    "COC(=O)C[C@H](N)C(=O)N1CC[C@H](C(=O)OC)[C@H](C)C1",
    "C[C@H](NC[C@@H]1[C@@H]2CCC[C@@H]21)C(=O)N1CCCCC1",
    "CCN(CC(F)(F)F)C(=O)[C@@H]1CCCN(C(=O)[C@H](C)C(C)C)C1",
    "CO[C@@H](C)C(=O)N1CCC(C)(C(N)=S)CC1",
    "CN(C)C[C@@H](N)COCCC(=O)N1CCCCC1",
    "CON(CC1CC1)C(=O)N1CC[C@@H](C(N)=S)C1",
    "CC(C)N1CCN(C2CN(C(=O)[C@@]3(N)C[C@@H]3C)C2)CC1",
    "CCNC(=O)CC(=O)O[C@@H](C)C(=O)N1CCC[C@H](C(=O)OCC)C1",
    "CCN(CCC(N)=S)[C@@H](C)C(=O)N1CCCCCC1",
    "CC(C)[C@@H](O)CN1CC[C@H](C(=O)N(C)C)C1",
    "CC(C)C[C@@H](N)CC(=O)N1CCOC[C@@H](CO)C1",
    "C[C@H](N)[C@@H](C)C(=O)CC1CCCC1",
    "CCCC[C@H](C)N(CCCC)C(=O)[C@H]1CCCN(C(=O)[C@@H](C)CC)C1",
    "C=C(C)CCNC[C@@H](C)N(C)C(=O)[C@@H]1CCCN(C(C)=O)C1",
    "CCC(CC)CC(=O)O[C@@H](C(=O)N1CCC[C@@H](C(N)=O)C1)C(C)C",
    "C[C@@H](C(=O)N1CCN(C(=O)C(=O)N2CCCC2)CC1)C(F)(F)F",
    "CCC(CC)[S@@+]([O-])[C@@H](C)C(=O)N1CCC(C)CC1"
]

This block converts the list of molecule sequences to JSON, calls the MolMIM hidden-state endpoint, extracts the `hiddens` array, and converts it to a Pandas DataFrame for clustering.


In [ ]:
# Convert the sequences to a JSON string
data = json.dumps({"sequences": molecules})

# Make the API call to get the hidden-state embeddings
response = requests.post(url, headers=headers, data=data, timeout=60)
response.raise_for_status()

# Parse the response as JSON
embeddings = response.json()["hiddens"]

# Convert the embeddings to a Pandas DataFrame
df = pd.DataFrame(np.squeeze(np.array(embeddings)))


This final block plots a hierarchically clustered heatmap using seaborn. The heatmap shows the similarity between the molecule embeddings, with similar molecules clustered together. The x-axis shows the embedding dimension and the y axis shows the index of the molecule in the original list.

In [ ]:
# Plot the hierarchically clustered heatmap using seaborn
sns.set()
plt.figure(figsize=(8, 8))
sns.clustermap(df, method="ward", cmap="coolwarm", col_cluster=False)
plt.show()

Run the previous plotting cell to render the clustered heatmap. Similar molecules should appear closer together in the dendrogram and heatmap pattern.